# Thompson Sampling for non-stationary armed bandits

**Author:** François Cinotti

---

**Thompson Sampling** is a powerful tool for solving "armed-bandit tasks", which is widely used in online advertising and recommendation systems. Its strength lies in efficiently balancing exploration and exploitation, starting with very exploratory choices that allow the algorithm to learn, before driving it to almost exclusive exploitation of the best option. However, when faced with non-stationary bandits, standard Thompson Sampling performs poorly as it cannot easily shift back to an exploratory phase. In this tutorial, I will show two methods of adressing this weakness: **Dynamic Thompson Sampling (Gupta et al. 2011)**, and **Sliding-Window Thompson Sampling (Trovo et al. 2020)** , which I used for comparison with animal behaviour in a paper dealing with animal meta-learning (Cinotti et al. 2024).

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta as beta_dist

# Fixed random seed for reproducibility, remove to see different outcomes
np.random.seed(42)

# Use a clean plot style
plt.style.use('seaborn-v0_8-whitegrid')

---
## Part 1: The Beta Distribution

### Basic properties
Suppose I gave you a coin and you wanted to check it was a fair coin. You would probably toss it a few times, count how many heads and tails you got, and make your mind up depending on whether the two numbers were roughly the same. But how many times should you toss it and what does "roughly the same" mean? The **beta distribution** can help you answer these questions rigorously, if you're a hard-boiled mathematician, or better yet, visually, if you're a mere mortal. The beta distribution is a probability density function defined for values between 0 and 1 that typically has a more or less broad skewed bell shape. This is not the place to delve into its formula and properties, and for all intents and purposes, all we really need to know is that it has two **shape parameters** called **$\alpha$**, which pulls the peak of the function towards 1, and **$\beta$** (an admittedly confusing choice of name) which pulls the peak towards 0. Indeed, the location of the peak, or mode of the distribution, which can loosely be understood as the most likely value when sampling the distribution, is ${\alpha -1} \over {\alpha + \beta - 2}$. Meanwhile, the sum aggregate of $\alpha$ and $\beta$ narrows the peak. Because probabilities themselves lie between 0 and 1, the beta distribution lends itself well to representing belief about a probability itself, a probability of a probability if you like. Mathematically, "*lends itself well*" is a bit of an undersell, the beta distribution is the conjugate prior of binomial probabilities (probabilities for binary outcomes such as coin tosses), it has properties, that go beyond the scope of this notebook, that make it especially ideal for representing belief about a binomial probability. Conveniently for us, this function and different methods linked to it come ready-built in the Python library *scipy.stats* (see the import above).

### Prior beliefs
To make the most of beta distributions in our coin flipping test, you must first decide on a **prior** belief, an initial beta distribution with parameters $\alpha_0$ and $\beta_0$. A nice choice, reflecting how much you trust me, would be a beta distribution that is already centred on 0.5, let's say $\alpha_0 = \beta_0 = 5$. This would be quite an informative prior as the shape of the function is already a pretty well defined bell. On the other hand, I'm just a guy on the internet you've never met, so you are understandably very suspicious and you expect the probability of heads to be very close to either 0 or 1, which you could achieve by setting $\alpha_0=\beta_0=1/2$ (this is in fact quite a special choice known as *Jeffreys's prior*, with the cool property of being invariant to reparameterisation, i.e. if instead of the probability of heads, you were thinking in terms of odds ratios, this prior would guarantee that switching between the two doesn't matter). A third possibility, which is the one we will use later in Thompson sampling, is an agnostic flat belief or **uniform prior**: the probability of heads could be anywhere between 0 and 1; this is obtained by setting $\alpha_0=\beta_0=1$.


In [ ]:
# Plot different prior beliefs

x = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('A selection of prior beliefs', fontsize=13)

# informative prior
axes[0].plot(x, beta_dist.pdf(x, 5, 5), 'k', lw = 2)
axes[0].set_ylabel('Density')
axes[0].set_xlabel('Probability of heads')
axes[0].set_title('Trusting prior')

# Jeffreys prior
axes[1].plot(x, beta_dist.pdf(x, 0.5, 0.5), 'k', lw = 2)
axes[1].set_xlabel('Probability of heads')
axes[1].set_title('Suspicious prior')

# uniform prior
axes[2].plot(x, beta_dist.pdf(x, 1, 1), 'k', lw=2)
axes[2].set_xlabel('Probability of heads')
axes[2].set_title('Agnostic prior')
axes[2].set_ylim([0,2])

plt.tight_layout()
plt.show()

### Updating prior beliefs

You've chosen a prior, now let's imagine you ran two different experiments: in the first you flipped the coin 10 times and got 6 heads and 4 tails which probably sounds fair, and in the second you flipped it 100 times and observed 60 heads versus 40 tails. Notice that the ratio of heads over tails is the same between the two experiments, what changes is the total number of tosses made in each experiment. To represent your belief about this coin's probability of heads after both experiments, you simply plug in the number of heads into the value of $\alpha$ and conversely the number of tails into $\beta$:

- $\alpha = \alpha_0 + n_{heads}$
- $\beta = \beta_0 + n_{tails}$

In [ ]:
# Plot posterior beliefs after experiment 1 for different prior beliefs
x = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Posterior beliefs', fontsize=13)

# informative prior
axes[0].plot(x, beta_dist.pdf(x, 5+6, 5+4), 'steelblue', lw = 2, label = 'after 10 flips')
axes[0].plot(x, beta_dist.pdf(x, 5+60, 5+40), 'tomato', lw = 2, label = 'after 100 flips')
axes[0].set_ylabel('Density')
axes[0].set_xlabel('Probability of heads')
axes[0].set_title('Trusting prior')
axes[0].legend()

# Jeffreys prior
axes[1].plot(x, beta_dist.pdf(x, 0.5+6, 0.5+4), 'steelblue', lw = 2, label = 'after 10 flips')
axes[1].plot(x, beta_dist.pdf(x, 0.5+60, 0.5+40), 'tomato', lw = 2, label = 'after 100 flips')
axes[1].set_xlabel('Probability of heads')
axes[1].set_title('Suspicious prior')
axes[1].legend()

# uniform prior
axes[2].plot(x, beta_dist.pdf(x, 1+6, 1+4), 'steelblue', lw=2, label = 'after 10 flips')
axes[2].plot(x, beta_dist.pdf(x, 1+60, 1+40), 'tomato', lw=2, label = 'after 100 flips')
axes[2].set_xlabel('Probability of heads')
axes[2].set_title('Agnostic prior')
axes[2].legend()

plt.tight_layout()
plt.show()

Several lessons are worth pointing out here. Firstly, the number of observations really does matter. Not so much for the position of the peaks, which, with the exception of the trusting prior, is not visibly different between the two experiments (remember the mode is equal to ${\alpha -1} \over {\alpha + \beta - 2}$ which would be exactly the same between the two experiments if it weren't for the small initial parameters) but it does matter for the narrowness or variance of the distribution. If you stopped at 10 flips, then a real probability of heads of 0.5 is entirely believable, as is a wide range of other values from 0.3 to 0.8... On the other hand, after 100 trials, despite the same ratio of heads, 0.5 is starting to look like it's on the edges of likely values which range somewhere between 0.5 and 0.7. Secondly, contrary to what critics of **Bayesian statistics** might argue (yes, you have unwittingly been following a beginners' course to Bayesian statistics), the choice of the prior quickly loses any meaningful impact on the final outcome. Even with just 10 observations, the three distribution functions overlap considerably for the three different priors, and even more so after 100 observations.

---
## Part 2: Multi-armed bandits

Now forget coin flipping, time for the big league; imagine you are in a casino facing three slot machines ("arms"). Each machine pays out a reward with some unknown probability. Obviously, your goal is to maximise your total reward over a series of trials.

This is the classic **exploration-exploitation dilemma**:
- **Exploit**: keep pulling the arm you currently believe is best
- **Explore**: try other arms in case one of them is actually better

A purely greedy strategy (always exploit) risks getting stuck on a suboptimal arm while a purely random strategy (always explore) wastes trials on bad arms. What you want is a strategy that appropriately balances this **exploration-exploitation trade-off**: explore when you're uncertain and exploit when you know which is the best. This is the task **Thompson sampling** is designed for.

In the next sections, we will work with an example of a **three-armed bandit** with the following reward probabilities:

| Arm | Reward probability |
|-----|-------------------|
| 1   | 0.6 ✓ (best)      |
| 2   | 0.2               |
| 3   | 0.2               |

On each trial, pulling arm $i$ yields a reward of 1 with probability $p_i$, and 0 otherwise.



In [ ]:
# Define the true reward probabilities for each arm
# The agent does NOT have access to these — they are the ground truth we use to simulate the task
TRUE_PROBS = np.array([0.6, 0.2, 0.2])
n_arms = len(TRUE_PROBS)

def pull_arm(arm, probs):
    """
    Simulate pulling an arm.
    Returns 1 (reward) with probability probs[arm], else 0.
    """
    return int(np.random.rand() < probs[arm])

---
## Part 3: Thompson Sampling

### The key idea

Instead of storing a single estimate of each arm's reward probability, Thompson Sampling represents them using beta distributions that paint a full picture of all possible values of each arm's reward probability. To take full advantage of this rich information, deciding which lever to choose on each trial depends on random **samples** from each distribution and picking the arm that yielded the biggest sample. At the beginning, when the prior distributions are still very broad (typically, a uniform prior, the convention I will go with for the remainder of this notebook), any arm might give a large sample so a lot of exploration can take place. As observations accumulate however, the beta distributions sharpen and shift apart so that exploitation of the best lever increases gradually over time.


### The algorithm

More precisely, there are four steps on every trial:

1. **Sampling** a value from each arm
2. **Selecting** arm i with the highest sampled value
3. **Observing** the reward (0 or 1)
4. **Updating** the beta distribution of the chosen arm:
   - If reward = 1: $\alpha_i \leftarrow \alpha_i + 1$
   - If reward = 0: $\beta_i \leftarrow \beta_i + 1$

Now let's run Thompson sampling on our three-armed bandit and see how well it identifies each arm:

In [ ]:
def thompson_sampling(n_trials, prob_schedule):
    """
    Run vanilla Thompson Sampling on a bandit task.
    
    Parameters
    ----------
    n_trials : Number of trials to run
    prob_schedule : array of shape (n_trials, n_arms)
        True reward probabilities at each trial
    
    Returns
    -------
    choices : Which arm was chosen on each trial
    rewards : Reward received on each trial
    alphas : Evolution of alpha parameters over trials
    betas : Evolution of beta parameters over trials
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype=int)
    rewards = np.zeros(n_trials, dtype=int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):
        
        # step 1: sample one value from each arm's beta distribution
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2: choose the arm with the highest sample
        choice = np.argmax(samples)
        
        # step 3: pull the arm and observe the reward
        reward = pull_arm(choice, prob_schedule[t]) # output = 1 or 0
        
        # step 4: update the Beta distribution of the chosen arm
        alpha[choice] += reward
        beta[choice] += 1 - reward
        
        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run Thompson Sampling for 500 trials
N_TRIALS = 500
prob_schedule = np.tile(TRUE_PROBS, (N_TRIALS, 1))
choices, rewards, alphas, betas = thompson_sampling(N_TRIALS, prob_schedule)

print(f"Final arm selection counts: Arm 1: {np.sum(choices==0)}, "
      f"Arm 2: {np.sum(choices==1)}, Arm 3: {np.sum(choices==2)}")

### Results

The final arm count tells us that arm 1 was indeed selected a vast majority of times with some exploration of the two other arms. We will now plot the time course of this experiment (the two cells are kept deliberately separate to allow you to modify plotting parameters without writing over simulation results): 

In [ ]:
# ---------------- User parameters to play with -----------------
window = 20 # size of smoothing window for performance
snapshot_T = 500 # trial at which to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
colors = ['steelblue', 'tomato', 'green']

# ---------------------------------------------------------------
# LEFT FIGURE: PERFORMANCE + RASTER PLOT BELOW

# Compute the proportion of trials on which the best arm was chosen

best_arm_chosen = (choices == 0).astype(float)
performance = np.convolve(best_arm_chosen, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes[0].plot(performance, color='steelblue', lw=2)
axes[0].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes[0].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('Trial')
axes[0].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes[0].set_title('Performance')
axes[0].legend()
axes[0].set_ylim(-bottom_margin, 1.05)
axes[0].set_xlim(0, len(choices))

# horizontal separator between curve area and raster
axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes[0].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices == arm_idx)[0]
    axes[0].vlines(
            trial_indices,
            y_center - tick_height * 0.5,
            y_center + tick_height * 0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes[0].text(
            0 - 25, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# ---------------------------------------------------------------
# RIGHTT FIGURE: SNAPSHOT OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)

for i in range(n_arms):
    a, b = alphas[snapshot_T-1, i], betas[snapshot_T-1, i]
    axes[1].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={TRUE_PROBS[i]})')
    axes[1].axvline(TRUE_PROBS[i], color=colors[i], linestyle='--', alpha=0.5)
axes[1].set_xlabel('Reward probability')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Beta distributions after {snapshot_T} trials')
axes[1].legend()

plt.tight_layout()
plt.show()

Details of the following interpretation may depend on the simulation. After about 100 trials of exploration, the agent chooses arm 1 almost exclusively. Notice how at the end of the experiment, after 500 trials, the beta distribution of arm 1 is quite narrow and close to the real value of 0.6, while there is still a lot of uncertainty about arm 2 and arm 3 especially. This is because, after an initial phase of exploration, the overlap between the three distributions became so small that selecting either of these two other arms was extremely unlikely, hence their distributions remain quite uncertain. At this point, further reducing this uncertainty doesn't actually matter, all the agent cares about is the fact they are definitely worse than arm 1. On the other hand, because the algorithm keeps selecting arm 1 and it doesn't stop learning, the beta distribution for arm 1 has become very narrow as evidence concerning this arm accumulates, with consequences we shall explore in the next section.

---
## Part 4: Non-Stationary Multi-Armed Bandits

### The problem

In the real world — and in many neuroscience experiments — reward probabilities change over time. In the rat experiment from Cinotti et al. (2024) for example, the identity of the best lever changed every 24 trials without warning.

The key issue with vanilla Thompson Sampling is that it accumulates evidence indefinitely. As we have just seen, after many trials, the beta distributions become very narrow and confident, and when the reward probabilities switch, the agent is slow to notice because it takes many new observations to meaningfully shift a distribution that is already very concentrated. Furthermore, even if the distribution does gradually shift, it does not ever broaden again, and in the long run, the distribution represents an average across the entire past so that, if, as in the next example, the correct arm rotates between the different options, they eventually all converge towards the same distribution, preventing any useful discrimination of the current best option.

### Our non-stationary task

To illustrate this issue, we will now simulate a task where the reward probabilities rotate every 500 trials and run Thompson sampling:

| Block | Arm 1 | Arm 2 | Arm 3 |
|-------|-------|-------|-------|
| 1     | **0.6** | 0.2 | 0.2 |
| 2     | 0.2 | **0.6** | 0.2 |
| 3     | 0.2 | 0.2 | **0.6** |
| 4     | **0.6** | 0.2 | 0.2 |
| ...   | ... | ... | ... |

In [ ]:
def make_nonstationary_schedule(n_trials, block_size=500):
    """
    Generate a schedule of reward probabilities that rotate every block_size trials.
    The best arm rotates: 1 -> 2 -> 3 -> 1 -> ...
    
    Returns
    -------
    prob_schedule : array of shape (n_trials, n_arms)
        True reward probabilities at each trial
    best_arm_schedule : array of int
        Which arm is best at each trial
    """
    prob_schedule = np.zeros((n_trials, n_arms))
    best_arm_schedule = np.zeros(n_trials, dtype=int)
    
    for t in range(n_trials):
        # Which block are we in?
        block = (t // block_size) % n_arms
        
        # The best arm rotates with each block
        probs = [0.2, 0.2, 0.2]
        probs[block] = 0.6
        
        prob_schedule[t] = probs
        best_arm_schedule[t] = block
    
    return prob_schedule, best_arm_schedule

# Generate the schedule
N_TRIALS_NS = 3000  # 6 blocks of 100 trials
BLOCK_SIZE = 500
prob_schedule, best_arm_schedule = make_nonstationary_schedule(N_TRIALS_NS, BLOCK_SIZE)

print("Reward probabilities in each block:")
for b in range(6):
    t = b * BLOCK_SIZE
    print(f"  Block {b+1} (trials {t+1}-{t+BLOCK_SIZE}): → best arm: {best_arm_schedule[t]+1}")
print("=" * 80)

# Run vanilla Thompson Sampling on the non-stationary task
choices, rewards, alphas, betas = thompson_sampling(N_TRIALS_NS, prob_schedule)

print(f"Final arm selection counts: Arm 1: {np.sum(choices==0)}, "
      f"Arm 2: {np.sum(choices==1)}, Arm 3: {np.sum(choices==2)}")

### Results

Since the identity of the best arm rotates between the three arms for equal numbers of trials, a good decision-making algorithm should select each arm roughly the same amount of times. From the final arm count, we can see this is not the case: arm 1 has been chosen a disproportionately large amount of times compared to the other two, probably because it has the advantage of having been the correct arm in the first block.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameters to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)
colors = ['steelblue', 'tomato', 'green']

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title('Vanilla Thompson sampling in a non-stationary environment\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices))

# separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height * 0.5,
            y_center + tick_height * 0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 80, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas[trial-1, i], betas[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

Vanilla Thompson Sampling performs well in the first block — it very quickly identifies the best arm and selects it almost all the time. But after the block change, performance drops and recovers more slowly than it had originally risen at the start of block 1, displaying a decrease in flexibility. 

**The reason:** by the end of the first block, arm 1 has accumulated many successes. Its beta distribution is narrow and centred close to 0.6. Even after the task switches to block 2 and arm 1 starts returning fewer rewards, it takes many trials of failure to shift this very confident distribution leftward. You can see how 200 trials into block 2 (trial 700), the peak of the distribution of arm 1 is still much further right than that of the correct arm 2 which, in addition, is still very broad, making it even more likely to sample smaller values than arm 1. In block 3, performance actually improves a lot faster than it did in block 2, although not as fast as in block 1; this is thanks to the combination of two factors: first the beta distribution of arm 1 has already had 500 trials of block 2 to decay (move to the left), while the distribution of arm 2 has not been able to grow as much as arm 1 had by the end of block 1 because of poor performance in block 2. Block 4 is a repeat of block 1, and despite having already experienced this block, performance of Thompson sampling is worse than in block 1, because competing against arms 2 and 3, whose distributions are now considerably closer to 0.6, has become more difficult. Similar interpretations can be made for blocks 5 and 6 which are repeats of blocks 2 and 3. At the end of the experiment (trial 3000), all three distributions overlap considerably, making selection very noisy.

This motivates two extensions designed specifically for non-stationary environments.

---
## Part 5: Dealing with Non-Stationary Environments

### 5.1 Sliding-Window Thompson Sampling (SWTS)

Proposed by Trovo et al. (2020), SWTS takes a very straightforward approach to solve this issue. Fundamentally, Thompson sampling underperforms because it remembers too well: old outcomes count as much in shaping the beta distributions as recent ones (contrary to Q-learning, which weighs outcomes based on how recent they were, a topic we might explore another time) so that it takes ever greater amounts of evidence to overturn the accumulated evidence. What you need is a filtering process to ensure that recent outcomes count more. As its name suggests, the *sliding-window* approach takes a radical route by only counting observations over the last T trials. Outcomes in that window all have the same weight, but at least they are not being accumulated since the very start of the experiment. The size of the sliding-window, T, controls how flexible the algorithm is: if it's small, the algorithm learns quickly but is noisy, if it's large it adapts more slowly but is more stable.

In [ ]:
def sliding_window_thompson_sampling(n_trials, prob_schedule, T):
    """
    Sliding Window Thompson Sampling (Trovo et al., 2020).
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype = int)
    rewards = np.zeros(n_trials, dtype = int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):

        # step 1: Sample
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2 : select arm that gave the largest sample
        choice = np.argmax(samples)

        # step 3 : pull arm and observe
        reward = pull_arm(choice, prob_schedule[t])
        
        # step 4: update the Beta distribution of the chosen arm
        alpha[choice] += reward
        beta[choice] += 1 - reward

        # step 5: forget the last trial outside of window size 
        if t >= T: # the sliding-window only starts operating after T trials
            edge = t - T # the last trial to slip out of the window
            past_choice = choices[edge]
            past_rwd = rewards[edge]

            if past_rwd == 1:
                alpha[past_choice] -= 1
            else:
                beta[past_choice] -= 1

        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run SWTS on the non-stationary task
T = 100
choices_SWTS, rewards_SWTS, alphas_SWTS, betas_SWTS = sliding_window_thompson_sampling(N_TRIALS_NS, prob_schedule, T)

print(f"Final arm selection counts: Arm 1: {np.sum(choices_SWTS==0)}, "
      f"Arm 2: {np.sum(choices_SWTS==1)}, Arm 3: {np.sum(choices_SWTS==2)}")

This time, the final arm count shows much less bias in arm selection counts.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameter to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices_SWTS == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title(f'Sliding-Window Thompson sampling in a non-stationary environment (T = {T})\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices_SWTS))

# horizontal separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices_SWTS == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height*0.5,
            y_center + tick_height*0.5,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 50, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas_SWTS[trial-1, i], betas_SWTS[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

While performance is never as stable as it was in block 1 for vanilla Thompson sampling, SWTS shifts between arms according to the probability schedule correctly. Indeed, the boundaries of block changes can be guessed from the raster plot which clearly depicts rotating periods of arm dominance. The noisiness of arm selection is due to the underlying beta distributions which remain very broad even at the end of blocks. For instance, after 3000 trials, the algorithm has come to an end of 500 trials of a block where arm 3 was the best lever, and while the distribution of arm 3 is indeed very close to 0.6, the two competing arms have such broad distributions that selection of these suboptimal options still occurs relatively often.


### 5.2 Dynamic Thompson Sampling (DTS)

Although older than SWTS, Dynamic Thompson Sampling, introduced in Gupta et al. (2011), adopts a more sophisticated approach. Instead of ignoring older trials, DTS focuses on the beta distributions themselves, constraining them to have a minimum broadness. To implement this, recall how the broadness of the beta distributions depends on $\alpha + \beta$: the larger the sum, the more narrow the peak is around the mean. So, to ensure a minimum broadness, or variance in technical terms, at each trial, after observing the outcome of selecting arm i, DTS checks that $\alpha_i + \beta_i < C$, with C a pre-defined parameter. If the check is passed, then the $\alpha_i$ and $\beta_i$ are updated as usual. If not, then the update rule becomes:
   - If reward = 1: $\alpha_i \leftarrow (\alpha_i + 1) * C / (C+1)$; $\beta_i \leftarrow \beta_i * C / (C+1)$
   - If reward = 0: $\beta_i \leftarrow (\beta_i + 1) * C/(C+1)$; $\alpha_i \leftarrow \alpha_i * C / (C+1)$
     
You can easily check for yourself that this modification ensures that $\alpha_i + \beta_i <= C$. It's worth noticing how, contrary to vanilla Thompson sampling, both shape parameters are updated simultaneously according to this update rule. Another point of difference, is that, instead of a global constraint on the total number of past observations, the check is made separately for each arm, so not selecting an arm for a long time does not mean its distribution decays back to uniform making it more likely to be tested.

Mathematically, this update guarantees that the variance is constrained. In fact, for the more mathematically inclined, it's probably worth thinking in terms of what you want your minimal variance to be, then working your way to the value of C. Another interesting mathematical property of this update, which you can see demonstrated in the original paper of Gupta et al. (2011), is that it implicitly assigns exponentially decaying weights to past outcomes so that the more recent ones are more heavily weighted than old ones. 


In [ ]:
def dynamic_thompson_sampling(n_trials, prob_schedule, C):
    """
    Dynamic Thompson Sampling (Gupta et al., 2011).
    """
    
    # initialise beta distribution parameters
    alpha = np.ones(n_arms)
    beta = np.ones(n_arms)
    
    # trial-by-trial storage of results
    choices = np.zeros(n_trials, dtype = int)
    rewards = np.zeros(n_trials, dtype = int)
    alphas = np.zeros((n_trials, n_arms))
    betas = np.zeros((n_trials, n_arms))
    
    for t in range(n_trials):

        # step 1: Sample
        samples = [np.random.beta(alpha[i], beta[i]) for i in range(n_arms)]
        
        # step 2 : select arm that gave the largest sample
        choice = np.argmax(samples)

        # step 3 : pull arm and observe
        reward = pull_arm(choice, prob_schedule[t])
        
        # step 4: update the Beta distribution of the chosen arm
        if alpha[choice] + beta[choice] < C:
            alpha[choice] += reward 
            beta[choice] += 1 - reward
        else:
            alpha[choice] = (alpha[choice] + reward) * C / (C+1) 
            beta[choice] = (beta[choice] + 1 - reward) * C / (C+1)

        # store results of this trial
        choices[t] = choice
        rewards[t] = reward
        alphas[t] = alpha
        betas[t] = beta
    
    return choices, rewards, alphas, betas

# Run DTS on the non-stationary task
C = 20
choices_DTS, rewards_DTS, alphas_DTS, betas_DTS = dynamic_thompson_sampling(N_TRIALS_NS, prob_schedule, C)

print(f"Final arm selection counts: Arm 1: {np.sum(choices_DTS==0)}, "
      f"Arm 2: {np.sum(choices_DTS==1)}, Arm 3: {np.sum(choices_DTS==2)}")

As for SWTS, the arm selection counts are roughly the same, suggestive of good choice flexibility.

In [ ]:
# Separate cell for plotting without relaunching simulation

# ---------------- User parameter to play with ------------------
window = 20 # size of smoothing window for performance
snapshots = [700, 3000] # two snapshots to view beta distribution
# ---------------------------------------------------------------

fig, axes = plt.subplot_mosaic(
    [['perf', 'perf'],
     ['beta1', 'beta2']],
    figsize=(14, 8)
)

# ----------------------------------------------------------------
# TOP FIGURE: PERFORMANCE + RASTER

correct = (choices_DTS == best_arm_schedule).astype(float)
performance = np.convolve(correct, np.ones(window)/window, mode='valid')

# How much vertical space to reserve below y=0 for the tick rows
tick_height = 0.06           # height of each tick-row band in data units
bottom_margin = tick_height * 3 + 0.02

axes['perf'].plot(performance, color='steelblue', lw=2)
axes['perf'].axhline(0.33, color='gray', linestyle='--', label='Chance (1/3)')
axes['perf'].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Perfect')
axes['perf'].set_xlabel('Trial')
axes['perf'].set_ylabel(f'Proportion choosing best arm\n(running average over {window} trials)')
axes['perf'].set_title(f'Dynamic Thompson sampling in a non-stationary environment (C = {C})\n'
                        'Red dotted lines = block changes (reward probabilities switch)')
axes['perf'].legend()
axes['perf'].set_ylim(-bottom_margin, 1.05)
axes['perf'].set_xlim(0, len(choices_DTS))

# horizontal separator line between curve area and ticks
axes['perf'].axhline(0, color='gray', linewidth=0.8, linestyle='-', alpha=0.4)

# Raster plot

for arm_idx in range(n_arms):
# vertical centre of this arm's tick row (negative = below the curve)
    y_center = -(arm_idx + 0.5) * tick_height

    # draw a faint background stripe so rows are easy to distinguish
    axes['perf'].axhspan(
            y_center - tick_height / 2,
            y_center + tick_height / 2,
            color=colors[arm_idx], alpha=0.07, linewidth=0
        )

    # one tick per trial on which this arm was chosen
    trial_indices = np.where(choices_DTS == arm_idx)[0]
    axes['perf'].vlines(
            trial_indices,
            y_center - tick_height * 0.35,
            y_center + tick_height * 0.35,
            color=colors[arm_idx], linewidth=0.5, alpha=0.6
        )

    # label on the right margin
    axes['perf'].text(
            0 - 50, y_center,
            f'Arm {arm_idx + 1}',
            va='center', ha='left',
            fontsize=8, color=colors[arm_idx]
        )

# Mark block boundaries
for b in range(1, N_TRIALS_NS // BLOCK_SIZE):
    axes['perf'].axvline(b * BLOCK_SIZE, color='red', linestyle=':', alpha=0.7)
    axes['perf'].text(b * BLOCK_SIZE + 2, 0.95, f'Block {b+1}', color='red', fontsize=8)

# ------------------------------------------------------------------
# BOTTOM FIGURES : TWO SNAPSHOTS OF BETA DISTRIBUTIONS

x = np.linspace(0, 1, 200)
colors = ['steelblue', 'tomato', 'green']

for ax_key, trial in zip(['beta1', 'beta2'], snapshots):
    for i in range(n_arms):
        a, b = alphas_DTS[trial-1, i], betas_DTS[trial-1, i]
        axes[ax_key].plot(x, beta_dist.pdf(x, a, b), color=colors[i], lw=2,
                 label=f'Arm {i+1} (true p={prob_schedule[trial-1,i]})')
        axes[ax_key].axvline(prob_schedule[trial-1,i], color=colors[i], linestyle='--', alpha=0.5)
    axes[ax_key].set_xlabel('Reward probability')
    axes[ax_key].set_ylabel('Density')
    axes[ax_key].set_title(f'Beta distributions after {trial} trials')
    axes[ax_key].legend()    

plt.tight_layout()
plt.show()

Selection of arm 1 in block 1 may not be quite as exclusive as for vanilla Thompson sampling but it is very high, consistent, and far less noisy than SWTS. Similar to SWTS, transition between blocks is handled smoothly, performance dropping and recovering very quickly. The underlying beta distributions at trial 3000 show how broad these are while remaining clearly separate, contrary to SWTS, meaning they are both discriminate enough to avoid further exploration but readily updatable in case of a block reversal. This is partly thanks to the fact the reward probabilities were chosen to be so different: if they had been closer, e.g. 0.4 versus 0.6, a higher value of C might have proved necessary. This brings us to a broader reflection about parameters. In these examples, T and C were chosen by hand, so that comparing the two algorithms comes with a big asterisk: are these differences systematic or can they be mitigated by different choices. Furthermore, these choices also impact performance with regard to the test environment. To ensure performance in a noisier environment with more frequent block rotations would require tuning the parameters, an issue that is non-existent for vanilla Thompson sampling.

---

## References

- N. Gupta, O. -C. Granmo and A. Agrawala, "Thompson Sampling for Dynamic Multi-armed Bandits," 2011 10th International Conference on Machine Learning and Applications and Workshops, Honolulu, HI, USA, 2011, pp. 484-489. Available at https://www.researchgate.net/publication/232616670_Thompson_Sampling_for_Dynamic_Multi-armed_Bandits
- Trovo, F., Paladino, S., Restelli, M., & Gatti, N. (2020). Sliding-window Thompson sampling for non-stationary settings.
Journal of Artificial Intelligence Research, 68, 311–364. Available at:. https://doi.org/10.1613/jair.1.11407
- Cinotti, F., Coutureau, E., Khamassi, M., Marchand, A. R., & Girard, B. (2024). Regulation of reinforcement learning parameters captures long-term changes in rat behaviour. European Journal of Neuroscience, 60(4), 4469–4490. https://doi.org/10.1111/ejn.16449

